# Evaluating the Ideal Chunk Size for a RAG System using LlamaIndex

# **Introduction**

Retrieval-augmented generation (RAG) has introduced an innovative approach that fuses the extensive retrieval capabilities of search systems with the LLM. When implementing a RAG system, one critical parameter that governs the system’s efficiency and performance is the `chunk_size`. How does one discern the optimal chunk size for seamless retrieval? This is where LlamaIndex `Response Evaluation` comes handy. In this blogpost, we'll guide you through the steps to determine the best `chunk size` using LlamaIndex’s `Response Evaluation` module. If you're unfamiliar with the `Response` Evaluation module, we recommend reviewing its [documentation](https://docs.llamaindex.ai/en/latest/core_modules/supporting_modules/evaluation/modules.html) before proceeding.

## **Why Chunk Size Matters**

Choosing the right `chunk_size` is a critical decision that can influence the efficiency and accuracy of a RAG system in several ways:

1. **Relevance and Granularity**: A small `chunk_size`, like 128, yields more granular chunks. This granularity, however, presents a risk: vital information might not be among the top retrieved chunks, especially if the `similarity_top_k` setting is as restrictive as 2. Conversely, a chunk size of 512 is likely to encompass all necessary information within the top chunks, ensuring that answers to queries are readily available. To navigate this, we employ the Faithfulness and Relevancy metrics. These measure the absence of ‘hallucinations’ and the ‘relevancy’ of responses based on the query and the retrieved contexts respectively.
2. **Response Generation Time**: As the `chunk_size` increases, so does the volume of information directed into the LLM to generate an answer. While this can ensure a more comprehensive context, it might also slow down the system. Ensuring that the added depth doesn't compromise the system's responsiveness is crucial.

In essence, determining the optimal `chunk_size` is about striking a balance: capturing all essential information without sacrificing speed. It's vital to undergo thorough testing with various sizes to find a configuration that suits the specific use-case and dataset.

## **Setup**

Before embarking on the experiment, we need to ensure all requisite modules are imported:

In [ ]:
!pip install llama-index pypdf

INFO: pip is looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.2/328.2 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 115.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.3/303.3 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from llama_index.core import (
    SimpleDirectoryReader,
    VectorStoreIndex,
    Settings,
)

from llama_index.core.evaluation import (
    DatasetGenerator,
    FaithfulnessEvaluator,
    RelevancyEvaluator,
)

from llama_index.llms.openai import OpenAI

import openai
import time

## **Download Data**

We'll be using the Uber 10K SEC Filings for 2021 for this experiment.

In [ ]:
!mkdir -p 'data/10k/'
!wget 'https://raw.githubusercontent.com/jerryjliu/llama_index/main/docs/examples/data/10k/uber_2021.pdf' -O 'data/10k/uber_2021.pdf'

--2025-12-21 06:37:49--  https://raw.githubusercontent.com/jerryjliu/llama_index/main/docs/examples/data/10k/uber_2021.pdf
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1880483 (1.8M) [application/octet-stream]
Saving to: ‘data/10k/uber_2021.pdf’

data/10k/uber_2021. 100%[===================>]   1.79M  8.36MB/s    in 0.2s    

2025-12-21 06:37:50 (8.36 MB/s) - ‘data/10k/uber_2021.pdf’ saved [1880483/1880483]



## **Load Data**

Let’s load our document.

In [ ]:
# Load Data

reader = SimpleDirectoryReader("./data/10k/")
documents = reader.load_data()

## **Question Generation**

To select the right `chunk_size`, we'll compute metrics like Average Response time, Faithfulness, and Relevancy for various `chunk_sizes`. The `DatasetGenerator` will help us generate questions from the documents.

In [ ]:
from llama_index.core.evaluation import (
    DatasetGenerator,
    FaithfulnessEvaluator,
    RelevancyEvaluator,
)

eval_documents = documents[:20]  # pick first 20 docs
data_generator = DatasetGenerator.from_documents(eval_documents)  # pass documents
eval_questions = data_generator.generate_questions_from_nodes(num=40)


/usr/local/lib/python3.12/dist-packages/llama_index/core/evaluation/dataset_generation.py:201: DeprecationWarning: Call to deprecated class DatasetGenerator. (Deprecated in favor of `RagDatasetGenerator` which should be used instead.)
  return cls(
/usr/local/lib/python3.12/dist-packages/llama_index/core/evaluation/dataset_generation.py:297: DeprecationWarning: Call to deprecated class QueryResponseDataset. (Deprecated in favor of `LabelledRagDataset` which should be used instead.)
  return QueryResponseDataset(queries=queries, responses=responses_dict)


## Setting Up Evaluators

We are setting up the GPT-4 model to serve as the backbone for evaluating the responses generated during the experiment. Two evaluators, `FaithfulnessEvaluator` and `RelevancyEvaluator`, are initialised with the `service_context` .

1. **Faithfulness Evaluator** - It is useful for measuring if the response was hallucinated and measures if the response from a query engine matches any source nodes.
2. **Relevancy Evaluator** - It is useful for measuring if the query was actually answered by the response and measures if the response + source nodes match the query.

In [ ]:
from llama_index.core import Settings
from llama_index.llms.openai import OpenAI
from llama_index.core.evaluation import FaithfulnessEvaluator, RelevancyEvaluator

original_llm = Settings.llm

Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0)

faithfulness_gpt4 = FaithfulnessEvaluator()
relevancy_gpt4 = RelevancyEvaluator()

Settings.llm = original_llm


## **Response Evaluation For A Chunk Size**

We evaluate each chunk_size based on 3 metrics.

1. Average Response Time.
2. Average Faithfulness.
3. Average Relevancy.

Here's a function, `evaluate_response_time_and_accuracy`, that does just that which has:

1. VectorIndex Creation.
2. Building the Query Engine**.**
3. Metrics Calculation.

In [ ]:
from llama_index.core import Settings, VectorStoreIndex
from llama_index.llms.openai import OpenAI
import time

def evaluate_response_time_and_accuracy(chunk_size, eval_questions):
    total_response_time = 0
    total_faithfulness = 0
    total_relevancy = 0

    # --- Save original settings ---
    original_llm = Settings.llm
    original_chunk_size = Settings.chunk_size

    # --- Generation LLM (cheap & fast) ---
    Settings.llm = OpenAI(
        model="gpt-4o-mini",
        temperature=0
    )
    Settings.chunk_size = chunk_size

    # --- Build index ---
    vector_index = VectorStoreIndex.from_documents(eval_documents)
    query_engine = vector_index.as_query_engine()

    num_questions = len(eval_questions)

    for question in eval_questions:
        start_time = time.time()
        response = query_engine.query(question)
        elapsed_time = time.time() - start_time

        # --- Switch to evaluator LLM ---
        Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0)

        faithfulness_result = faithfulness_gpt4.evaluate_response(
            response=response
        ).passing

        relevancy_result = relevancy_gpt4.evaluate_response(
            query=question,
            response=response
        ).passing

        total_response_time += elapsed_time
        total_faithfulness += faithfulness_result
        total_relevancy += relevancy_result

        # --- Switch back to generation LLM ---
        Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0)

    # --- Restore original settings ---
    Settings.llm = original_llm
    Settings.chunk_size = original_chunk_size

    return (
        total_response_time / num_questions,
        total_faithfulness / num_questions,
        total_relevancy / num_questions,
    )


## **Testing Across Different Chunk Sizes**

We'll evaluate a range of chunk sizes to identify which offers the most promising metrics

In [ ]:
eval_questions = [
    "What is the main topic of the document?",
    "Summarize the key ideas discussed.",
    "What conclusions does the document reach?",
    "What evidence is provided to support the claims?",
    "Who is the intended audience of the document?"
]
for chunk_size in [128, 256, 512, 1024, 2048]:
    avg_response_time, avg_faithfulness, avg_relevancy = (
        evaluate_response_time_and_accuracy(chunk_size, eval_questions)
    )

    print(
        f"Chunk size {chunk_size} | "
        f"Avg Response Time: {avg_response_time:.2f}s | "
        f"Avg Faithfulness: {avg_faithfulness:.2f} | "
        f"Avg Relevancy: {avg_relevancy:.2f}"
    )


Chunk size 128 | Avg Response Time: 2.30s | Avg Faithfulness: 0.80 | Avg Relevancy: 0.80
Chunk size 256 | Avg Response Time: 2.05s | Avg Faithfulness: 0.80 | Avg Relevancy: 1.00
Chunk size 512 | Avg Response Time: 2.21s | Avg Faithfulness: 1.00 | Avg Relevancy: 1.00
Chunk size 1024 | Avg Response Time: 2.72s | Avg Faithfulness: 1.00 | Avg Relevancy: 1.00
Chunk size 2048 | Avg Response Time: 2.61s | Avg Faithfulness: 0.80 | Avg Relevancy: 1.00
